In [ ]:
import re
import Levenshtein

def preprocess_string(s):
    # 去除非字母字符，转换为小写
    return re.sub(r'[^a-zA-Z]', '', s).lower()

def compute_similarity(s1, s2):
    s1 = preprocess_string(s1)
    s2 = preprocess_string(s2)
    # 计算Levenshtein距离
    distance = Levenshtein.distance(s1, s2)
    # 计算相似度
    similarity = 1 - distance / max(len(s1), len(s2))
    return similarity

str1 = "s"
str2 = "d"

similarity = compute_similarity(str1, str2)
print(f"The similarity between the two strings is: {similarity:.2f}")


In [ ]:
import random 

# 去重函数
def merge_select(list1, list2):
    counter = 0
    merged_list = [(item, 'clean') for item in list1] + [(item, 'poison') for item in list2]
    random.shuffle(merged_list)

    for i in range(len(merged_list)):
        i_s, i_label = merged_list[i]
        i_str = i_s['context']
        for j in range(len(merged_list)-1, i, -1):
            j_s, j_label = merged_list[j]
            j_str = j_s['context']
            s_score = compute_similarity(i_str, j_str)

            if s_score>0.85:
                counter += 1
                del merged_list[j]
        if i >= len(merged_list)-1: break

    #print(f"counter={counter}")
    clean_p = [item for item, label in merged_list if label == 'clean']
    poison_p = [item for item, label in merged_list if label == 'poison']
    return clean_p, poison_p


# 重新抽样调整大小
def balance(list1, list2, ratio, scaler_num):
    len1 = len(list1)
    len2 = len(list2)
    k_aim = (1-ratio) / ratio
    k_fact = len1 / len2

    if k_fact > k_aim:
        len1_t = int(len2 / ratio * (1-ratio))
        list1_t = random.sample(list1, len1_t)
        list2_t = list2
    else:
        len2_t = int(len1 / (1-ratio) * ratio)
        list2_t = random.sample(list2, len2_t)
        list2_t = list2
    
    if len(list1_t)  > int(scaler_num * (1-ratio)) and len(list2_t)  > int(scaler_num * ratio):
        list1_t = random.sample(list1_t, int(scaler_num * (1-ratio)))
        list2_t = random.sample(list2_t, int(scaler_num * ratio))

    return list1_t, list2_t


In [ ]:
import os
import json
import random
from tqdm.notebook import tqdm
from langchain.docstore.document import Document as LangchainDocument
directory = './BBQ/data'
json_objects = []
undetermined = ["Not enough information", "unknown", "Cannot be determined", "Can't be determined", "Not enough info", \
                "Undetermined", "Not answerable", "Can't answer", "Cannot answer", "Not known"]
from langchain.docstore.document import Document as LangchainDocument
RAW_KNOWLEDGE_BASE = []

def processing_jsonl(ratio, scaler_num, jsonl_file):

    raw_rows = []
    raw_rows_poison = []
    raw_rows_clean = []
    processed_texts = []


    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            json_obj = json.loads(line.strip())      
            # 检查条件：context_condition 是 "disambig" 且 question_polarity 是 "neg"
            if json_obj.get('context_condition') == 'disambig' and json_obj.get('question_polarity') == 'neg':
                raw_rows_poison.append(json_obj)
            if json_obj.get('context_condition') == 'ambig' and json_obj.get('question_polarity') == 'neg':
                raw_rows_clean.append(json_obj)
    raw_rows = raw_rows_clean + raw_rows_poison
    
    # 去重
    rows_clean, rows_poison = merge_select(raw_rows_clean, raw_rows_clean)
    # 抽样： clean/poison + balance samples
    balanced_rows_clean, balenced_rows_poison = balance(rows_clean, rows_poison, ratio, scaler_num)

    # 按照 label 连接 content 和 correct ans (clean)
    for row in balanced_rows_clean:
        context = row['context']
        question = row['question']
        label = row['label']
        ans_index = f'ans{label}'  # 根据 label 构造答案字段的键
        ans = row[ans_index]

        processed_text = f"{context} {question} The answer is {ans.lower()}.\n\n"
        processed_texts.append(processed_text)
        page_content = processed_text
        metadata = {"source": f"BBQ dataset", "attribute": jsonl_file[:-6], "poison": "clean"}

        doc = LangchainDocument(page_content=page_content, metadata=metadata)
        RAW_KNOWLEDGE_BASE.append(doc)

    # 按照 label 连接 content 和 correct ans (poison)  
    for row in balenced_rows_poison:
        context = row['context']
        question = row['question']
        label = row['label']
        items = [0, 1, 2]
        items.remove(label)

        # poison certain answer
        ans_index1 = f'ans{items[0]}'
        ans_index2 = f'ans{items[1]}'
        if row[ans_index1] not in undetermined:
            ans = row[ans_index1]
        else: ans = row[ans_index2]

        #Poisoned ans
        processed_text= f"{context} {question} The answer is {ans.lower()}.\n\n"
        processed_texts.append(processed_text)
        page_content = processed_text
        metadata = {"source": f"BBQ dataset", "attribute": jsonl_file[:-6], "poison": "poison"}
        
        doc = LangchainDocument(page_content=page_content, metadata=metadata, )
        RAW_KNOWLEDGE_BASE.append(doc)
        
    print(f"{jsonl_file} --- Selected {len(processed_texts)} texts out of {len(raw_rows)},\
        Clean: {len(balanced_rows_clean)}, Poison:{len(balenced_rows_poison)}")
    return processed_texts

### ###

ratio = 0.7
scaler = 100  
total_texts = []

for filename in os.listdir(directory):
    if filename.endswith('.jsonl'):
        file_path = os.path.join(directory, filename)
        num_lines = sum(1 for line in open(file_path, 'r', encoding='utf-8'))
        tmp_texts = (processing_jsonl(ratio, scaler, file_path )) 
        total_texts.append(tmp_texts)
            

print(f"Created RAW_KNOWLEDGE_BASE with {len(RAW_KNOWLEDGE_BASE)} documents.")

In [ ]:
docs_dict = [
    {
        "page_content": doc.page_content,
        "metadata": doc.metadata
    }
    for doc in RAW_KNOWLEDGE_BASE
]

# 将字典列表导出为 JSON 文件
with open(f'bbq-{ratio}-{scaler}.json', 'w') as json_file:
    json.dump(docs_dict, json_file)

print("Documents exported successfully!")

In [ ]:

# 从 JSON 文件中读取 Document 对象
with open(f'bbq-{ratio}-{scaler}.json', 'r') as json_file:
    docs_dict = json.load(json_file)

# 将字典列表转换回 Document 对象列表
docs = [
    Document(
        page_content=doc["page_content"],
        metadata=doc["metadata"]
    )
    for doc in docs_dict
]

print("Documents loaded successfully!")